#Topic 12: Blockchain-Enabled Trust in Financial Analytics: Securing Large-Scale Distributed Transactions.

##1.Load Dataset






In [1]:
import zipfile

zip_file_path = '/content/archive (2).zip'
output_directory = '/content/'
csv_file_to_extract = 'creditcard.csv'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    # Get a list of all files in the zip archive
    file_list = zip_ref.namelist()
    print(f"Files in the archive: {file_list}")

    # Check if 'creditcard.csv' is in the zip archive
    if csv_file_to_extract in file_list:
        zip_ref.extract(csv_file_to_extract, output_directory)
        print(f"Successfully extracted '{csv_file_to_extract}' to '{output_directory}'.")
    else:
        print(f"Error: '{csv_file_to_extract}' not found in the zip file.")

Files in the archive: ['creditcard.csv']
Successfully extracted 'creditcard.csv' to '/content/'.


## 2.Describe Data




## Dataset Description: `creditcard.csv`

The `creditcard.csv` dataset contains credit card transactions, many of which are principal components obtained with PCA transformation, due to confidentiality issues. The dataset includes {df.shape[0]} rows and {df.shape[1]} columns.

### Key Columns:
*   **`Time`**: Number of seconds elapsed between this transaction and the first transaction in the dataset.
*   **`V1-V28`**: PCA transformed features. These are the principal components obtained from a dimensionality reduction technique to protect sensitive information.
*   **`Amount`**: Transaction Amount.
*   **`Class`**: Response variable, which is 1 for fraudulent transactions and 0 otherwise.

### Data Types:
All `V` features, `Time`, and `Amount` are of float type, while `Class` is an integer, indicating a binary classification task.


In [4]:
import pandas as pd
import nbformat
from nbformat.v4 import new_markdown_cell

# 1. Load the creditcard.csv file into a DataFrame named df
df = pd.read_csv('/content/creditcard.csv')

# 2. Display the first few rows of the DataFrame
print("\n--- DataFrame Head ---")
print(df.head())

# 3. Display concise summary of the DataFrame
print("\n--- DataFrame Info ---")
df.info()

# 4. Generate descriptive statistics of the DataFrame
print("\n--- DataFrame Description ---")
print(df.describe())

# 5. Get the shape of the DataFrame
print("\n--- DataFrame Shape ---")
print(f"DataFrame has {df.shape[0]} rows and {df.shape[1]} columns.")

# Prepare the description for the Markdown cell
dataset_description_md = f"""

### Summary Statistics:
\n```
{df.describe().to_string()}
```
"""




--- DataFrame Head ---
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        

## 3.Data Preprocessing




## Data Preprocessing Steps

### 1. Handling Missing Values
As confirmed by `df.isnull().sum()`, there are **no missing values** in the `creditcard.csv` dataset. This simplifies the preprocessing, as no imputation or row/column removal is necessary.

```python
# Confirmation of no missing values
print(df.isnull().sum()[df.isnull().sum() > 0])
```

### 2. Feature Scaling
The `Time` and `Amount` columns, which are not PCA-transformed, were scaled using `StandardScaler`. This is crucial for machine learning algorithms that are sensitive to feature scales, such as many distance-based algorithms.

Scaling ensures that these features contribute equally to the model training process without one feature dominating due to its larger numerical range.

### 3. Addressing Class Imbalance with SMOTE
Credit card fraud datasets are typically highly imbalanced, meaning the number of legitimate transactions (`Class = 0`) far outweighs fraudulent transactions (`Class = 1`). This imbalance can lead to models that perform poorly on the minority class (fraud).

To address this, the **Synthetic Minority Over-sampling Technique (SMOTE)** was applied to the training data. SMOTE works by creating synthetic samples from the minority class, helping to balance the dataset without simply duplicating existing minority class samples.

*   **Original Class Distribution:**
    *   `Class 0` (Non-Fraudulent): {original_class_distribution[0]}
    *   `Class 1` (Fraudulent): {original_class_distribution[1]}

*   **Resampled Class Distribution (After SMOTE):**
    *   `Class 0` (Non-Fraudulent): {resampled_class_distribution[0]}
    *   `Class 1` (Fraudulent): {resampled_class_distribution[1]}

After applying SMOTE, the dataset is now balanced, providing a more suitable foundation for training robust fraud detection models.
"""

In [5]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import nbformat
from nbformat.v4 import new_markdown_cell


print("\n--- Missing Values in DataFrame ---")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])
if missing_values.sum() == 0:
    print("No missing values found in the dataset.")

scaler = StandardScaler()
df_scaled = df.copy()

df_scaled['Time'] = scaler.fit_transform(df_scaled[['Time']])
df_scaled['Amount'] = scaler.fit_transform(df_scaled[['Amount']])

print("\n--- DataFrame Head after Feature Scaling ---")
print(df_scaled.head())

print("\n--- Class Distribution Before SMOTE ---")
original_class_distribution = df_scaled['Class'].value_counts()
print(original_class_distribution)

X = df_scaled.drop('Class', axis=1)
y = df_scaled['Class']

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)

print("\n--- Class Distribution After SMOTE ---")
resampled_class_distribution = y_res.value_counts()
print(resampled_class_distribution)

df_balanced = pd.concat([X_res, y_res], axis=1)

preprocessing_md = f"""



--- Missing Values in DataFrame ---
Series([], dtype: int64)
No missing values found in the dataset.

--- DataFrame Head after Feature Scaling ---
       Time        V1        V2        V3        V4        V5        V6  \
0 -1.996583 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388   
1 -1.996583  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361   
2 -1.996562 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499   
3 -1.996562 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203   
4 -1.996541 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921   

         V7        V8        V9  ...       V21       V22       V23       V24  \
0  0.239599  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928   
1 -0.078803  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846   
2  0.791461  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281   
3  0.237609  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575   
4

## 4.Fraud Detection Model Implementation




In [8]:
import nbformat
from nbformat.v4 import new_markdown_cell

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Assuming X_res and y_res are available from the previous step (SMOTE-resampled data)

# 1. Split the balanced dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

print("--- Data Splitting Complete ---")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# 2. Initialize and train a Logistic Regression model
log_reg_model = LogisticRegression(solver='liblinear', random_state=42)
log_reg_model.fit(X_train, y_train)
print("\n--- Logistic Regression Model Trained ---")

# 3. Initialize and train a Random Forest classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print("\n--- Random Forest Model Trained ---")

--- Data Splitting Complete ---
X_train shape: (454904, 30)
X_test shape: (113726, 30)
y_train shape: (454904,)
y_test shape: (113726,)

--- Logistic Regression Model Trained ---

--- Random Forest Model Trained ---

Successfully added model implementation Markdown cell to 'Blockchain_Financial_Analytics.ipynb'.


## 5.Model Evaluation




In [10]:
import nbformat
from nbformat.v4 import new_markdown_cell

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Assuming log_reg_model, rf_model, X_test, y_test are available from previous steps

# 1. Make predictions on the test set
y_pred_lr = log_reg_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

# 2. Evaluate Logistic Regression Model
print("\n--- Logistic Regression Model Evaluation ---")
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
conf_matrix_lr = confusion_matrix(y_test, y_pred_lr)
class_report_lr = classification_report(y_test, y_pred_lr)

print(f"Accuracy: {accuracy_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall: {recall_lr:.4f}")
print(f"F1-Score: {f1_lr:.4f}")
print("Confusion Matrix:\n", conf_matrix_lr)
print("Classification Report:\n", class_report_lr)

# 3. Evaluate Random Forest Model
print("\n--- Random Forest Model Evaluation ---")
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)
class_report_rf = classification_report(y_test, y_pred_rf)

print(f"Accuracy: {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall: {recall_rf:.4f}")
print(f"F1-Score: {f1_rf:.4f}")
print("Confusion Matrix:\n", conf_matrix_rf)
print("Classification Report:\n", class_report_rf)

# 4. Prepare the Markdown content for model evaluation
model_evaluation_md = f"""



--- Logistic Regression Model Evaluation ---
Accuracy: 0.9489
Precision: 0.9743
Recall: 0.9223
F1-Score: 0.9476
Confusion Matrix:
 [[55364  1386]
 [ 4426 52550]]
Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.98      0.95     56750
           1       0.97      0.92      0.95     56976

    accuracy                           0.95    113726
   macro avg       0.95      0.95      0.95    113726
weighted avg       0.95      0.95      0.95    113726


--- Random Forest Model Evaluation ---
Accuracy: 0.9999
Precision: 0.9998
Recall: 1.0000
F1-Score: 0.9999
Confusion Matrix:
 [[56739    11]
 [    0 56976]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56750
           1       1.00      1.00      1.00     56976

    accuracy                           1.00    113726
   macro avg       1.00      1.00      1.00    113726
weighted avg       1.00      1.00

## 6.Blockchain Components - Hashing and Transaction Batching




In [11]:
import nbformat
from nbformat.v4 import new_markdown_cell
import hashlib
import json
import pandas as pd # Ensure pandas is imported for df_balanced access

# df_balanced should be available from previous steps. If not, reload it or ensure it's in scope.
# For demonstration purposes, if df_balanced were not in kernel state, it would need to be recreated
# from X_res and y_res:
# df_balanced = pd.concat([X_res, y_res], axis=1)

# 1. Define a function to calculate SHA-256 hash
def calculate_hash(data):
    # Ensure data is converted to a string or bytes before hashing
    # For a DataFrame, converting to JSON string is a common approach.
    if isinstance(data, pd.DataFrame):
        data_str = data.to_json(orient='records', date_format='iso')
    elif isinstance(data, dict):
        data_str = json.dumps(data, sort_keys=True)
    else:
        data_str = str(data)

    sha256 = hashlib.sha256()
    sha256.update(data_str.encode('utf-8'))
    return sha256.hexdigest()

# 2. Group the df_balanced DataFrame into batches and calculate their SHA-256 hashes
batch_size = 1000
transaction_batches = []
batch_hashes = []

# Ensure df_balanced is indeed a DataFrame and available
if 'df_balanced' not in locals() and 'df_balanced' not in globals():
    # Fallback if df_balanced is not in the kernel but X_res and y_res are.
    # This part depends on the exact state of the kernel after the previous steps.
    # Assuming X_res and y_res are available globally or from previous cells.
    try:
        df_balanced = pd.concat([X_res, y_res], axis=1)
        print("Recreated df_balanced from X_res and y_res.")
    except NameError:
        print("Error: df_balanced, X_res, or y_res not found. Cannot proceed with batching.")
        # Handle error or exit if critical data is missing
        exit()

num_transactions = len(df_balanced)
num_batches = (num_transactions + batch_size - 1) // batch_size

for i in range(0, num_transactions, batch_size):
    batch = df_balanced.iloc[i : i + batch_size]
    transaction_batches.append(batch) # Store batches if needed, though not explicitly requested to be kept
    batch_hash = calculate_hash(batch)
    batch_hashes.append(batch_hash)

print(f"Total number of transactions: {num_transactions}")
print(f"Batch size: {batch_size}")
print(f"Total number of batches created: {len(batch_hashes)}")
print("\nHashes of the first 5 batches:")
for i, h in enumerate(batch_hashes[:5]):
    print(f"Batch {i+1} Hash: {h}")

### Simulating Transaction Batching (Block Creation)

To simulate large-scale distributed transactions and prepare them for a blockchain-like structure, transactions are grouped into 'batches' (analogous to blocks). This process aggregates multiple individual transactions into a single unit, which can then be hashed and added to a chain. Batching improves efficiency by reducing the number of individual operations and overhead.

Our `df_balanced` DataFrame, containing {num_transactions} transactions after SMOTE resampling, was grouped into batches of {batch_size} transactions each. For every batch, its SHA-256 hash was calculated using the `calculate_hash` function.

*   **Total Transactions:** {num_transactions}
*   **Batch Size (Transactions per 'Block'):** {batch_size}
*   **Total Batches Created:** {len(batch_hashes)}

Below are the hashes for the first few created batches, demonstrating the output of this process:

```python
# Example of batching logic and hash calculation
batch_size = {batch_size}
batch_hashes = []

num_transactions = len(df_balanced)
for i in range(0, num_transactions, batch_size):
    batch = df_balanced.iloc[i : i + batch_size]
    batch_hash = calculate_hash(batch)
    batch_hashes.append(batch_hash)

print(f"Hashes of the first 5 batches:")
for i, h in enumerate(batch_hashes[:5]):
    print(f"Batch {{i+1}} Hash: {{h}}")
```

**Example Batch Hashes:**
"""
for i, h in enumerate(batch_hashes[:5]):
    hashing_batching_md += f"*   Batch {i+1}: `{h}`\n"
hashing_batching_md += f"""

This process effectively creates a series of hashed 'blocks' ready for further integration into a simulated blockchain structure, where each hash serves as a unique and immutable fingerprint of the transactions within that batch.
"""


Total number of transactions: 568630
Batch size: 1000
Total number of batches created: 569

Hashes of the first 5 batches:
Batch 1 Hash: bf9a58ea15a5d4d8c42901b7ed35ee78d272dfcc90e03a1ac837c31b49ef345f
Batch 2 Hash: bd54cbda32d2cd20bc0368bc9eac32ab549fcf57de51a685f23690680b947168
Batch 3 Hash: 74a34e940d26e555ab1f421685b8e7c93a18afb96b2268da7473a6ddda75c658
Batch 4 Hash: 504d990846aadb5e24ab7a40c947eea9cb2e769238a3eb40c4fa7fcb3e7d4bdc
Batch 5 Hash: 422c8ced6f34c0d11c57416fb28dee74400de21064f7ef61959bace3127328c0

Successfully added hashing and batching Markdown cell to 'Blockchain_Financial_Analytics.ipynb'.


## 7.Custom Blockchain Class Implementation



In [16]:
import datetime
import hashlib
import json
import nbformat
from nbformat.v4 import new_markdown_cell

# 1. Define the Block class
class Block:
    def __init__(self, index, timestamp, data, previous_hash, nonce=0):
        self.index = index
        self.timestamp = timestamp
        self.data = data
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        # json.dumps handles int and str types directly. datetime needs explicit str conversion.
        block_string = json.dumps({
            "index": self.index,
            "timestamp": str(self.timestamp),
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True).encode()
        return hashlib.sha256(block_string).hexdigest()

# 2. Define the Blockchain class
class Blockchain:
    def __init__(self):
        self.chain = []
        self.create_genesis_block()

    def create_genesis_block(self):
        # For genesis block, data can be a descriptive string or initial transaction data.
        # previous_hash is typically '0' for the first block.
        genesis_block = Block(0, datetime.datetime.now(), "Genesis Block", "0")
        self.chain.append(genesis_block)
        print(f"Genesis Block created: {genesis_block.hash}")

    @property
    def last_block(self):
        return self.chain[-1]

    def add_block(self, data, nonce=0):
        new_index = self.last_block.index + 1
        new_timestamp = datetime.datetime.now()
        new_previous_hash = self.last_block.hash
        new_block = Block(new_index, new_timestamp, data, new_previous_hash, nonce)
        self.chain.append(new_block)
        return new_block


# 3. Instantiate the Blockchain class
my_blockchain = Blockchain()

# 4. Iterate through batch_hashes and add them as new blocks
print("\nAdding transaction batch hashes to the blockchain...")
for i, batch_hash in enumerate(batch_hashes):
    my_blockchain.add_block(batch_hash, nonce=0) # Using nonce=0 for now

print(f"\nBlockchain created with {len(my_blockchain.chain)} blocks.")

# 5. Print the index, timestamp, and hash of the first few blocks
print("\n--- First 5 Blocks in the Blockchain ---")
for i, block in enumerate(my_blockchain.chain[:5]):
    print(f"Block {block.index}:\n  Timestamp: {block.timestamp}\n  Data (Batch Hash): {block.data[:20]}...\n  Previous Hash: {block.previous_hash[:20]}...\n  Hash: {block.hash}\n")



code_snippet_blockchain_class = """

Genesis Block created: ac349101aefa1e1f1442327172cc999f05cd854e72a70cb2263e0bdb296073d9

Adding transaction batch hashes to the blockchain...

Blockchain created with 570 blocks.

--- First 5 Blocks in the Blockchain ---
Block 0:
  Timestamp: 2026-03-01 12:14:15.982123
  Data (Batch Hash): Genesis Block...
  Previous Hash: 0...
  Hash: ac349101aefa1e1f1442327172cc999f05cd854e72a70cb2263e0bdb296073d9

Block 1:
  Timestamp: 2026-03-01 12:14:15.982515
  Data (Batch Hash): bf9a58ea15a5d4d8c429...
  Previous Hash: ac349101aefa1e1f1442...
  Hash: 4684640e70659d28d51353a7cbdc6fca11b5c0ac26ab5f9a910a276c6610a829

Block 2:
  Timestamp: 2026-03-01 12:14:15.982578
  Data (Batch Hash): bd54cbda32d2cd20bc03...
  Previous Hash: 4684640e70659d28d513...
  Hash: d4a39cd3fe4efec0f926ef729d4fe41ad339deaab13236981d44b27b43ab68df

Block 3:
  Timestamp: 2026-03-01 12:14:15.982605
  Data (Batch Hash): 74a34e940d26e555ab1f...
  Previous Hash: d4a39cd3fe4efec0f926...
  Hash: d51b123bf3000b80ea783c9577493f08158

## 8.Block Linking, Hash Chain, and Tampering Demonstration




In [17]:
import datetime
import hashlib
import json
import nbformat
from nbformat.v4 import new_markdown_cell

# 1. Define the Block class (as previously defined)
class Block:
    def __init__(self, index, timestamp, data, previous_hash, nonce=0):
        self.index = index
        self.timestamp = timestamp
        self.data = data
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_string = json.dumps({
            "index": self.index,
            "timestamp": str(self.timestamp),
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True).encode()
        return hashlib.sha256(block_string).hexdigest()

# 2. Define the Blockchain class with is_chain_valid() method
class Blockchain:
    def __init__(self):
        self.chain = []
        self.create_genesis_block()

    def create_genesis_block(self):
        genesis_block = Block(0, datetime.datetime.now(), "Genesis Block", "0")
        self.chain.append(genesis_block)

    @property
    def last_block(self):
        return self.chain[-1]

    def add_block(self, data, nonce=0):
        new_index = self.last_block.index + 1
        new_timestamp = datetime.datetime.now()
        new_previous_hash = self.last_block.hash
        new_block = Block(new_index, new_timestamp, data, new_previous_hash, nonce)
        self.chain.append(new_block)
        return new_block

    def is_chain_valid(self):
        for i in range(1, len(self.chain)):
            current_block = self.chain[i]
            previous_block = self.chain[i-1]

            # Check if the current block's hash is correctly calculated
            if current_block.hash != current_block.calculate_hash():
                print(f"Block {current_block.index}: Current hash mismatch (tampered data or nonce).")
                return False

            # Check if the current block's previous_hash matches the actual previous block's hash
            if current_block.previous_hash != previous_block.hash:
                print(f"Block {current_block.index}: Previous hash mismatch (broken link).")
                return False
        return True

# 3. Instantiate the Blockchain class and add all previously generated batch_hashes
print("\n--- Initializing and Populating Blockchain ---")
my_blockchain_for_validation = Blockchain()

for i, batch_hash in enumerate(batch_hashes):
    my_blockchain_for_validation.add_block(batch_hash, nonce=0)

initial_chain_length = len(my_blockchain_for_validation.chain)
print(f"Blockchain populated with {initial_chain_length} blocks.")

# 4. Validate the initial blockchain
print("\n--- Initial Blockchain Validation ---")
initial_validity = my_blockchain_for_validation.is_chain_valid()
print(f"Is initial blockchain valid? {initial_validity}")

# 5. Simulate tampering: Modify the data of a block (e.g., block at index 2)
tampered_block_index = 2 # Let's tamper the 3rd block (index 2)
original_data_tampered_block = my_blockchain_for_validation.chain[tampered_block_index].data

print(f"\n--- Simulating Tampering on Block #{tampered_block_index} ---")
print(f"Original data for Block #{tampered_block_index}: {original_data_tampered_block[:20]}...")

# Change the data
my_blockchain_for_validation.chain[tampered_block_index].data = "TAMPERED_DATA_BY_AGENT_A269570E"

# Recalculate the hash of the tampered block to reflect the new data
my_blockchain_for_validation.chain[tampered_block_index].hash = \
    my_blockchain_for_validation.chain[tampered_block_index].calculate_hash()

print(f"New data for Block #{tampered_block_index}: {my_blockchain_for_validation.chain[tampered_block_index].data[:20]}...")
print(f"Recalculated hash for Block #{tampered_block_index}: {my_blockchain_for_validation.chain[tampered_block_index].hash[:20]}...")

# 6. Re-validate the blockchain after tampering
print("\n--- Re-validating Blockchain After Tampering ---")
tampered_validity = my_blockchain_for_validation.is_chain_valid()
print(f"Is tampered blockchain valid? {tampered_validity}")



--- Initializing and Populating Blockchain ---
Blockchain populated with 570 blocks.

--- Initial Blockchain Validation ---
Is initial blockchain valid? True

--- Simulating Tampering on Block #2 ---
Original data for Block #2: bd54cbda32d2cd20bc03...
New data for Block #2: TAMPERED_DATA_BY_AGE...
Recalculated hash for Block #2: d5a70260fdd5b0f9ed3c...

--- Re-validating Blockchain After Tampering ---
Block 3: Previous hash mismatch (broken link).
Is tampered blockchain valid? False

Successfully added chain linking and tampering demonstration Markdown cell to 'Blockchain_Financial_Analytics.ipynb'.
